In [1]:
# =========================================
# CELL 1 — FULL SETUP + PATCH + INDENT FIX
# =========================================
import os
import re
import torch
from pathlib import Path

# 0. Fix cwd
os.chdir('/kaggle/working')
print("Current dir:", os.getcwd())

# 1. Clean + clone repo
!rm -rf Hunyuan3D-2
!git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git

# 2. Check GPU
if torch.cuda.is_available():
    print("✅ GPU:", torch.cuda.get_device_name(0))
    print("✅ VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")
else:
    print("❌ No GPU")

# 3. Enter project
os.chdir('/kaggle/working/Hunyuan3D-2')
print("Now in:", os.getcwd())

# 4. Install dependencies
!pip install -r requirements.txt
!pip install -e .
!npm install -g localtunnel

# 5. Build required modules
os.chdir('hy3dgen/texgen/custom_rasterizer')
!python setup.py install
os.chdir('../differentiable_renderer')
!python setup.py install
os.chdir('/kaggle/working/Hunyuan3D-2')

# 6. Patch Gradio for future-proof launch
gradio_file = 'gradio_app.py'
with open(gradio_file, 'r') as f:
    code = f.read()

# Simplify Blocks constructor
code = re.sub(
    r'with gr\.Blocks\(theme=.*?, title=.*?, analytics_enabled=.*?, css=.*?\) as demo',
    'with gr.Blocks(title="Hunyuan-3D-2.0", analytics_enabled=False) as demo',
    code
)

# Update demo.launch()
code = re.sub(
    r'demo\.launch\([^\)]*\)',
    'demo.launch(server_name="0.0.0.0", server_port=args.port, share=False)',
    code
)

# Fix file viewer bug
code = re.sub(
    r"relative_path = f'/static/\{os\.path\.relpath\(html_path, SAVE_DIR\)\}'",
    r"relative_path = f'/file={os.path.abspath(html_path)}'",
    code
)

with open(gradio_file, 'w') as f:
    f.write(code)

# 7. Fix indentation for SAVE_DIR + static_dir lines
with open(gradio_file, 'r') as f:
    lines = f.readlines()

fixed_lines = []
for line in lines:
    stripped = line.lstrip()
    if stripped.startswith('SAVE_DIR = args.cache_path') or \
       stripped.startswith('static_dir = Path(SAVE_DIR).absolute()') or \
       stripped.startswith('static_dir.mkdir('):
        fixed_lines.append('    ' + stripped)  # 4 spaces
    else:
        fixed_lines.append(line)

with open(gradio_file, 'w') as f:
    f.writelines(fixed_lines)

print("✅ Setup complete! Gradio patched and SAVE_DIR + static_dir indentation fixed.")

Current dir: /kaggle/working
Cloning into 'Hunyuan3D-2'...
remote: Enumerating objects: 1070, done.
remote: Total 1070 (delta 0), reused 0 (delta 0), pack-reused 1070 (from 1)
Receiving objects: 100% (1070/1070), 81.28 MiB | 40.46 MiB/s, done.
Resolving deltas: 100% (519/519), done.
✅ GPU: Tesla T4
✅ VRAM: 14.56 GB
Now in: /kaggle/working/Hunyuan3D-2
INFO: pip is looking at multiple versions of rembg to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.8/740.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.5/106.5 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 3.0 MB/s eta 0:00:00
Obtaining file:///kaggle/wo

In [2]:
# =========================================
# CELL 2 — LOCALTUNNEL + GRADIO (NON-BLOCKING + LOGS)
# =========================================
import subprocess
import threading
import random
import time

# Pick a free port
port = random.randint(7860, 7890)
print(f"⚡ Using port {port} for Gradio + LocalTunnel")

# -----------------------------
# 1️⃣ Start LocalTunnel in background
# -----------------------------
def start_localtunnel(port):
    process = subprocess.Popen(
        ['lt', '--port', str(port)],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        universal_newlines=True
    )
    url = None
    while True:
        line = process.stdout.readline()
        if line == '' and process.poll() is not None:
            break
        if line:
            line = line.strip()
            print("[LocalTunnel]", line)
            if line.startswith("https://"):
                url = line
                print("🚀 Your public URL is:", url)
                break
    return url

# Run LocalTunnel in a separate thread
lt_thread = threading.Thread(target=start_localtunnel, args=(port,), daemon=True)
lt_thread.start()

# Give LocalTunnel a few seconds to initialize
time.sleep(5)

# -----------------------------
# 2️⃣ Launch Hunyuan3D Gradio app with live logs
# -----------------------------
def launch_gradio():
    process = subprocess.Popen(
        [
            "python3", "gradio_app.py",
            "--model_path", "tencent/Hunyuan3D-2mv",
            "--subfolder", "hunyuan3d-dit-v2-mv",
            "--texgen_model_path", "tencent/Hunyuan3D-2",
            "--low_vram_mode",
            "--port", str(port)
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    for line in process.stdout:
        print("[Gradio]", line, end="")

# Launch Gradio in a separate thread
gr_thread = threading.Thread(target=launch_gradio, daemon=True)
gr_thread.start()

print("✅ Hunyuan3D is running! The LocalTunnel URL will appear above when ready.")
print("⚠️ Note: On Kaggle, you may need to complete the LocalTunnel IP verification page before opening the link.")

⚡ Using port 7863 for Gradio + LocalTunnel
[LocalTunnel] your url is: https://late-ants-pick.loca.lt
✅ Hunyuan3D is running! The LocalTunnel URL will appear above when ready.
⚠️ Note: On Kaggle, you may need to complete the LocalTunnel IP verification page before opening the link.
